La variable objetivo seleccionada para el modelado es Factura_Promedio_COP.

Las variables candidatas para el modelo son:

- Empresa
- Segmento
- Mes
- Año

In [174]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [175]:
df = pd.read_csv("sui_factura_promedio_consolidado.csv")

In [176]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.isnull().sum()

(5809, 6)
['Empresa', 'Anio', 'Mes', 'Periodo', 'Segmento', 'Factura_Promedio_COP']
Empresa                     str
Anio                      int64
Mes                       int64
Periodo                     str
Segmento                    str
Factura_Promedio_COP    float64
dtype: object


Empresa                 0
Anio                    0
Mes                     0
Periodo                 0
Segmento                0
Factura_Promedio_COP    0
dtype: int64

In [177]:
columnas = ['Anio', 'Mes', 'Segmento', 'Factura_Promedio_COP']

df = df[columnas]

Normalizar nombres: minúsculas, sin espacios

In [178]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [179]:
print(df)

      anio  mes              segmento  factura_promedio_cop
0     2025    1             Estrato 1          2.730945e+05
1     2025    1             Estrato 1          2.002508e+05
2     2025    1             Estrato 1          8.711506e+04
3     2025    1             Estrato 1          1.191730e+05
4     2025    1             Estrato 1          7.251058e+04
...    ...  ...                   ...                   ...
5804  2026    3  Total No Residencial          8.367119e+06
5805  2026    3  Total No Residencial          3.693736e+06
5806  2026    3  Total No Residencial          2.067820e+07
5807  2026    3  Total No Residencial          2.784656e+07
5808  2026    3  Total No Residencial          5.552995e+08

[5809 rows x 4 columns]


In [180]:
df.dtypes

anio                      int64
mes                       int64
segmento                    str
factura_promedio_cop    float64
dtype: object

In [181]:
df["factura_promedio_cop"] = (
    df["factura_promedio_cop"]
    .where(
        df["factura_promedio_cop"] >= 0,
        np.nan
    )
)

In [182]:
columns_category = df.select_dtypes(include=['object', 'string', 'category']).columns
print(columns_category)

Index(['segmento'], dtype='str')


In [183]:
df_model = pd.get_dummies(
    df,
    columns=columns_category,
    drop_first=True
)

In [184]:
df_model.dtypes.value_counts()

bool       11
int64       2
float64     1
Name: count, dtype: int64

In [185]:
df_model.dtypes

anio                               int64
mes                                int64
factura_promedio_cop             float64
segmento_Estrato 1                  bool
segmento_Estrato 2                  bool
segmento_Estrato 3                  bool
segmento_Estrato 4                  bool
segmento_Estrato 5                  bool
segmento_Estrato 6                  bool
segmento_Industrial                 bool
segmento_Oficial                    bool
segmento_Otros                      bool
segmento_Total No Residencial       bool
segmento_Total Residencial          bool
dtype: object

In [186]:
df_model['factura_promedio_cop'].describe()

count    5.808000e+03
mean     3.207198e+07
std      1.456971e+08
min      0.000000e+00
25%      1.171943e+05
50%      4.532722e+05
75%      3.407199e+06
max      1.969094e+09
Name: factura_promedio_cop, dtype: float64

In [187]:
print((df_model["factura_promedio_cop"] < 0).sum())

0


In [188]:
df_model['log_factura'] = np.log1p(df_model['factura_promedio_cop'])

60% train, 20% val, 20% test

In [189]:
df_train_full, df_test = train_test_split(df_model, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=42)

In [190]:
print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

Train: 3485, Val: 1162, Test: 1162


In [191]:
df_train.dtypes

anio                               int64
mes                                int64
factura_promedio_cop             float64
segmento_Estrato 1                  bool
segmento_Estrato 2                  bool
segmento_Estrato 3                  bool
segmento_Estrato 4                  bool
segmento_Estrato 5                  bool
segmento_Estrato 6                  bool
segmento_Industrial                 bool
segmento_Oficial                    bool
segmento_Otros                      bool
segmento_Total No Residencial       bool
segmento_Total Residencial          bool
log_factura                      float64
dtype: object

In [192]:
y_train = df_train['log_factura'].values
y_val = df_val['log_factura'].values
y_test = df_test['log_factura'].values

In [193]:
del df_train['log_factura'], df_train['factura_promedio_cop']
del df_val['log_factura'], df_val['factura_promedio_cop']
del df_test['log_factura'], df_test['factura_promedio_cop']

In [ ]:
from sklearn.linear_model import LinearRegression

In [195]:
X_train = df_train.values
X_val = df_val.values
X_test = df_test.values

print(X_train.shape)
print(y_train.shape)

(3485, 13)
(3485,)


In [196]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](13,)","[-0.26,-0.03,-2.82,..., 0.97, 0.6 ,-2.17]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,543.1
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,13
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](13,)","[211.19, 20.28, 19.57,..., 14.37, 14.2 , 5.38]"


In [197]:
y_pred = modelo.predict(X_val)

In [202]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)
print(f"Intercepto: {modelo.intercept_:.4f}")
print(f"Pesos: {modelo.coef_}")

Intercepto: 543.1491
Pesos: [-0.26094733 -0.031324   -2.82425002 -2.14828159 -2.49064832 -2.59852411
 -2.41524701 -2.12611215  1.52559418  0.64527295  0.96522283  0.59632814
 -2.17021856]
RMSE: 1.8701467434412025


In [207]:
mae = mean_absolute_error(y_val, y_pred)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

MAE: 1.2139
RMSE: 1.8701
R²: 0.4518


In [203]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_val)

In [204]:
mae_rf = mean_absolute_error(y_val, y_pred_rf)
rmse_rf = root_mean_squared_error(y_val, y_pred_rf)
r2_rf = r2_score(y_val, y_pred_rf)

print(mae_rf)
print(rmse_rf)
print(r2_rf)

1.293380308139
1.954208496041995
0.40144444959678016


In [205]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(
    random_state=42
)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_val)

In [206]:
mae_gb = mean_absolute_error(y_val, y_pred_gb)
rmse_gb = root_mean_squared_error(y_val, y_pred_gb)
r2_gb = r2_score(y_val, y_pred_gb)

print(mae_gb)
print(rmse_gb)
print(r2_gb)

1.2593416435630294
1.899308314244688
0.43460286046818364


In [208]:
resultados = pd.DataFrame({
    "Modelo": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        mae,
        mae_rf,
        mae_gb
    ],
    "RMSE": [
        rmse,
        rmse_rf,
        rmse_gb
    ],
    "R2": [
        r2,
        r2_rf,
        r2_gb
    ]
})

resultados

,Modelo,MAE,RMSE,R2
0,Linear Regression,1.213932,1.870147,0.451832
1,Random Forest,1.293380,1.954208,0.401444
2,Gradient Boosting,1.259342,1.899308,0.434603


MLflow

In [211]:
import mlflow

with mlflow.start_run(run_name="LinearRegression"):
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

2026/06/09 20:55:31 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/09 20:55:31 INFO mlflow.store.db.utils: Updating database tables


Artefactos

In [212]:
mlflow.sklearn.log_model(modelo, "model")

2026/06/09 20:56:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/09 20:56:43 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in E:\Datos_User\Desktop\UNIVERSIDAD\Especializacion Ingenieria de Software\SegundoSemestre\Nueva carpeta\proyectoaulaNE
2026/06/09 20:56:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/09 20:56:43 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in E:\Datos_User\Desktop\UNIVERSIDAD\Especializacion Ingenieria de Software\SegundoSemestre\Nueva carpeta\proyectoaulaNE
2026/06/09 20:56:43 INFO mlflow.utils.environm

Hiperparámetros Random Forest

In [210]:
RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o